In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
import operator
import requests

load_dotenv()

True

In [13]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    from urllib.parse import quote

    response = requests.get(
        f"https://wttr.in/{quote(location)}?format=j1",
        timeout=10,
    )
    response.raise_for_status()
    weather_data = response.json()
    current = weather_data["current_condition"][0]
    description = current["weatherDesc"][0]["value"]
    temperature = current["temp_C"]
    humidity = current["humidity"]
    return f"The current weather in {location} is {description} with a temperature of {temperature}°C and {humidity}% humidity."

In [14]:
@tool
def calculate(expression: str) -> str:
    """Calculate the result of a basic mathematical expression."""
    import ast

    binary_operators = {
        ast.Add: lambda left, right: left + right,
        ast.Sub: lambda left, right: left - right,
        ast.Mult: lambda left, right: left * right,
        ast.Div: lambda left, right: left / right,
        ast.Mod: lambda left, right: left % right,
        ast.Pow: lambda left, right: left ** right,
    }
    unary_operators = {
        ast.UAdd: lambda value: value,
        ast.USub: lambda value: -value,
    }

    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in binary_operators:
            return binary_operators[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in unary_operators:
            return unary_operators[type(node.op)](evaluate(node.operand))
        raise ValueError("Only numbers and basic arithmetic operators are allowed")

    try:
        parsed_expression = ast.parse(expression, mode="eval")
        return str(evaluate(parsed_expression.body))
    except (SyntaxError, ValueError, ZeroDivisionError, TypeError, OverflowError) as error:
        return f"Calculation error: {error}"

In [15]:
all_tools = [get_weather, calculate]

llm = ChatOpenAI(model="gpt-4o-mini")

class AgenticState(TypedDict):
    messages: Annotated[list, operator.add]

In [16]:
def agent_node(state: AgenticState) -> dict:
    llm_with_tools = llm.bind_tools(all_tools)
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgenticState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "__end__"

In [17]:
builder = StateGraph(AgenticState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(all_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tools", END])
builder.add_edge("tools", "agent")

memory = MemorySaver()
agent = builder.compile(checkpointer=memory)

In [19]:
query = "What was my last question?"
config = {"configurable": {"thread_id": "react-demo"}}
agent.invoke({"messages": [HumanMessage(content=query)]}, config=config)

{'messages': [HumanMessage(content='What is the weather in New York City and what is 5 + 3?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 85, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7f0cfe5367', 'id': 'chatcmpl-Emcoc1cbl4glw3O3IMDm3vcgQvIi1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'normalized_finish_reason': 'tool_call'}, id='lc_run--01a1fdb1-e895-73c2-ad93-63f23f35b16d-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York City'},